# 🔬 GIADA Task 3b — Diagnosi causale del trunk condiviso `m+h`
Un solo run development-only distingue supervisione dei rate, conflitto dei gradienti, warm-start e budget.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
WORK=Path('/kaggle/working/giada_task_3b'); GIADA_REPO=WORK/'giada'; TEACHER_REPO=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip(); print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
import torch
assert torch.cuda.is_available(),'La Task 3b preregistrata richiede una GPU CUDA Kaggle.'
from src.giada_teacher import ExtractedGateFormula,JointGateOptimizationDiagnosisConfig,prepare_joint_gate_dataset,run_joint_gate_optimization_diagnosis
prereg=json.loads((GIADA_REPO/'experiments/teacher_joint_gate_optimization_diagnosis_preregistration_v1.json').read_text())
task3=json.loads((GIADA_REPO/'experiments/teacher_joint_gate_cell_result_v1.json').read_text())
assert not task3['decision']['task4_authorized'] and prereg['fresh_accessed'] is False
display({'task3':task3['diagnosis'],'preregistration':prereg})


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_joint_m_h_optimization_diagnosis')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
bundle=prepare_joint_gate_dataset(formula); config=JointGateOptimizationDiagnosisConfig()
display({'contract':bundle['contract'],'arms':config.arms,'checkpoints':config.checkpoints,'baseline_extended_steps':config.baseline_extended_steps})


In [ ]:
report=run_joint_gate_optimization_diagnosis(bundle,OUTPUT_DIR,config,code_revision=REVISION)
display({'valid':report['valid'],'diagnosis':report['diagnosis'],'effects':report['causal_effect_fractions'],'supported':report['supported_causes'],'gradient_conflict':report['baseline_early_gradient_conflict_fraction'],'fresh_accessed':report['fresh_accessed'],'task4_authorized':report['task4_authorized'],'next_step':report['next_step']})
assert report['valid'] and not report['fresh_accessed'] and not report['task4_authorized']


## 📦 Download dell’artefatto
La cella seguente usa il download Blob/base64 compatibile con Kaggle.

In [ ]:
import base64
from IPython.display import Javascript, display
archive=Path(shutil.make_archive('/kaggle/working/giada_joint_m_h_optimization_diagnosis','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
